# Exploratory Data Analysis with SQL

The dataset is loaded into an SQLite database
(`data/spacex.db`, table `SPACEXTBL`) and explored with SQL queries.

In [1]:
import sqlite3, pandas as pd

df = pd.read_csv("data/spacex_launch_data.csv")
conn = sqlite3.connect("data/spacex.db")
df.to_sql("SPACEXTBL", conn, if_exists="replace", index=False)
print("SPACEXTBL loaded:", pd.read_sql("SELECT COUNT(*) AS n FROM SPACEXTBL", conn)["n"][0], "rows")

SPACEXTBL loaded: 90 rows

### 1. Unique launch sites

In [2]:
query = """SELECT DISTINCT LaunchSite FROM SPACEXTBL;"""
pd.read_sql(query, conn)

 LaunchSite
CCAFS LC-40
VAFB SLC-4E
 KSC LC-39A

### 2. First 5 launches from CCAFS

In [3]:
query = """SELECT * FROM SPACEXTBL WHERE LaunchSite LIKE 'CCA%' LIMIT 5;"""
pd.read_sql(query, conn)

 FlightNumber       Date BoosterVersion  PayloadMass Orbit  LaunchSite   Customer  Class       LandingOutcome
            1 2010-06-04        F9 v1.0          500   LEO CCAFS LC-40 Commercial      0           No attempt
            2 2010-12-08        F9 v1.0          544   LEO CCAFS LC-40 Commercial      0 Failure (drone ship)
            3 2012-05-22        F9 v1.0          525   LEO CCAFS LC-40 Commercial      0 Failure (drone ship)
            4 2012-10-08        F9 v1.0          500   LEO CCAFS LC-40 Commercial      0 Failure (drone ship)
            5 2013-03-01        F9 v1.0          677   LEO CCAFS LC-40 Commercial      0           No attempt

### 3. Total NASA payload (kg)

In [4]:
query = """SELECT SUM(PayloadMass) AS TotalNASA FROM SPACEXTBL WHERE Customer = 'NASA';"""
pd.read_sql(query, conn)

 TotalNASA
     45596

### 4. Average payload of F9 v1.1 (kg)

In [5]:
query = """SELECT AVG(PayloadMass) AS AvgV11 FROM SPACEXTBL WHERE BoosterVersion = 'F9 v1.1';"""
pd.read_sql(query, conn)

 AvgV11
 2928.4

### 5. First ground-pad landing

In [6]:
query = """SELECT MIN(Date) AS MinDate, LandingOutcome, LaunchSite FROM SPACEXTBL WHERE LandingOutcome LIKE 'Success (ground pad)';"""
pd.read_sql(query, conn)

   MinDate       LandingOutcome LaunchSite
2017-01-22 Success (ground pad) KSC LC-39A

### 6. Drone-ship successes, 4000-6000 kg

In [7]:
query = """SELECT BoosterVersion, PayloadMass, LandingOutcome FROM SPACEXTBL WHERE LandingOutcome = 'Success (drone ship)' AND PayloadMass BETWEEN 4000 AND 6000;"""
pd.read_sql(query, conn)

BoosterVersion  PayloadMass       LandingOutcome
         F9 FT         4088 Success (drone ship)
         F9 FT         5047 Success (drone ship)
         F9 B4         4973 Success (drone ship)
         F9 B5         4330 Success (drone ship)
         F9 B5         5954 Success (drone ship)
         F9 B5         4895 Success (drone ship)

### 7. Mission outcome counts

In [8]:
query = """SELECT CASE WHEN Class = 1 THEN 'Success' ELSE 'Failure' END AS Mission_Outcome, COUNT(*) AS COUNT FROM SPACEXTBL GROUP BY Class;"""
pd.read_sql(query, conn)

Mission_Outcome  COUNT
        Failure     29
        Success     61

### 8. Heaviest payload

In [9]:
query = """SELECT BoosterVersion, PayloadMass, Orbit FROM SPACEXTBL WHERE PayloadMass = (SELECT MAX(PayloadMass) FROM SPACEXTBL);"""
pd.read_sql(query, conn)

BoosterVersion  PayloadMass Orbit
         F9 B4        15600   GTO

### 9. 2015 drone-ship failures

In [10]:
query = """SELECT Date, BoosterVersion, LaunchSite, LandingOutcome FROM SPACEXTBL WHERE LandingOutcome = 'Failure (drone ship)' AND strftime('%Y', Date) = '2015';"""
pd.read_sql(query, conn)

      Date BoosterVersion  LaunchSite       LandingOutcome
2015-07-14        F9 v1.1 CCAFS LC-40 Failure (drone ship)
2015-09-30          F9 FT VAFB SLC-4E Failure (drone ship)
2015-11-04          F9 FT VAFB SLC-4E Failure (drone ship)
2015-12-28          F9 FT CCAFS LC-40 Failure (drone ship)

### 10. Landing outcome rank 2010-2017

In [11]:
query = """SELECT LandingOutcome, COUNT(*) AS COUNT FROM SPACEXTBL WHERE Date BETWEEN '2010-06-04' AND '2017-03-20' GROUP BY LandingOutcome ORDER BY COUNT DESC;"""
pd.read_sql(query, conn)

      LandingOutcome  COUNT
Failure (drone ship)     15
          No attempt     13
Success (drone ship)      3
Success (ground pad)      2